# Advanced 6D-MAN Scenarios

In [1]:
import sys
import os
# Navigate relative to the current working directory
sys.path.append(os.path.abspath('../../../src'))
import numpy as np
import pandas as pd
from sixdman.core.network import Network
from sixdman.core.band import Band, OpticalParameters
from sixdman.core.planning import PlanningTool
from sixdman.core.post_process import analyse_result
import networkx as nx
import json
from sixdman.utils.paths import get_project_root

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Use it to get paths
project_root = get_project_root()
results_dir = project_root / "results" / "MAN157_1000G_CAGR_0.4" / "NoBypass"
results_dir.mkdir(parents=True, exist_ok=True)

## *1. Create Network Instance*

In [3]:
# Initialize network
network = Network(topology_name = 'MAN157')

# Load topology from .mat file
network.load_topology(filepath = '../../../data/MAN157Nodes.mat', matrixName ='MAN157Nodes')

# Set hierarchical levels
hl_dict = network.define_hierarchy(
    HL1_standalone = [1, 5],
    HL2_standalone = [0, 2, 3, 4],
    HL3_standalone = list(range(6, 39)),
    HL4_standalone = list(range(39, 157))
)

## 2. *Define hierarchy node levels*

In [4]:
HL4_Standalone = hl_dict['HL4']['standalone']
HL4_colocated = hl_dict['HL4']['colocated']

HL3_Standalone = hl_dict['HL3']['standalone']
HL3_colocated = hl_dict['HL3']['colocated']

HL2_Standalone = hl_dict['HL2']['standalone']
HL2_colocated = hl_dict['HL2']['colocated']

HL1_Standalone = hl_dict['HL1']['standalone']

HL123_Standalone = HL1_Standalone + HL2_Standalone + HL3_Standalone
HL12_Standalone = HL1_Standalone + HL2_Standalone

arr = np.arange(network.adjacency_matrix.shape[0]) 
exclude_list = HL2_Standalone + HL3_Standalone + HL4_Standalone + HL1_Standalone
Optical_Nodes = arr[~np.isin(arr, exclude_list)]


## 3. Define Transmission Bands

In [5]:
# Define C-band parameters
c_band_params = OpticalParameters(Rs_mat = 170e9)

# Create C-band instance
c_band = Band(
    name='C',
    start_freq = 190.65, # THz
    end_freq = 196.675, # THz
    opt_params = c_band_params,
    network_instance = network,
    channel_spacing = 0.2 # THz
    )

# Define L-band parameters
l_band_params = OpticalParameters(Rs_mat = 170e9)

# Create L-band instance
l_band = Band(
    name='L',
    start_freq = 184.525, # THz
    end_freq = 190.565, # THz
    opt_params = l_band_params,
    network_instance = network,
    channel_spacing = 0.2 # THz
)

In [6]:
# define C-band and L-band frequency slots
spectrum_C = c_band.calc_spectrum()
spectrum_L = l_band.calc_spectrum()

# concatenate C-band and KL-band to a sigle frequency spectrum
spectrum = np.concatenate((spectrum_C, spectrum_L))

f_c_axis = spectrum * 1e12  # Convert to Hz
Pch_dBm = np.arange(-6, -0.9, 0.1)  # Channel power in dBm
num_Ch_mat = np.arange(1, len(spectrum) - 1)  # Channel indices

# define total number of frequency slots
num_fslots = len(num_Ch_mat)

In [7]:
C_band_num_channels = int(4800 / 200)
supC_band_num_channels = int(1200 / 200)
L_band_num_channels = int(6000 / 200)

## 4. Create Planning Tool and Optimize Network

In [8]:
# Initialize planning tool
planner = PlanningTool(
    network_instance = network,
    bands = [c_band, l_band], 
    optical_parameters = c_band_params,
    grid_center = f_c_axis, 
    period_time = 10)

all_node_degree = network.get_node_degrees(np.arange(network.adjacency_matrix.shape[0]))[:, 1]

In [9]:
minimum_hierarchy_level = 4
processing_level_list = [4, 3, 2]
splitter = []

In [10]:
for i in range(len(processing_level_list)):

    hierarchy_level = processing_level_list[i]

    print(f"Processing hierarchy level: {hierarchy_level}")

    match hierarchy_level:

        case 2:
            HL_Standalone = HL2_Standalone
            HL_colocated = HL2_colocated
            HL_lower_Standalone = HL4_Standalone + HL3_Standalone
            HL_up_Standalone = HL1_Standalone
            HL_all = np.concatenate((HL2_Standalone, HL2_colocated))
            capacity_updt_index = 0
            prev_hierarchy_level = 3
        case 3:
            HL_Standalone = HL3_Standalone
            HL_colocated = HL3_colocated
            HL_lower_Standalone = HL4_Standalone
            HL_up_Standalone = HL12_Standalone
            HL_all = np.concatenate((HL3_Standalone, HL3_colocated))
            capacity_updt_index = 1
            prev_hierarchy_level = 4
        case 4:
            HL_Standalone = HL4_Standalone
            HL_colocated = HL4_colocated
            HL_lower_Standalone = []
            HL_up_Standalone = HL123_Standalone
            HL_all = np.concatenate((HL4_Standalone, HL4_colocated))
            capacity_updt_index = 2
            prev_hierarchy_level = None

    sub_graph_HL, subnetMatrix_HL = network.compute_hierarchy_subgraph(hierarchy_level, minimum_hierarchy_level)
    HL_connected_nodes = network.get_neighbor_nodes(HL_Standalone) - set(HL_lower_Standalone)
    splitter.append(len(list(sub_graph_HL.edges())))

    file_name = results_dir/  f"{network.topology_name}_HL{hierarchy_level}_K_path_attributes.csv"

    if os.path.exists(file_name):

        print(f"Loading K-path attributes of HL{hierarchy_level} ...")

        K_path_attributes_df = pd.read_csv(file_name)
        K_path_attributes_df['links'] = K_path_attributes_df['links'].map(json.loads)
        K_path_attributes_df['nodes'] = K_path_attributes_df['nodes'].map(json.loads)
    else:

        print(f"Calculating K-path attributes of HL{hierarchy_level} ...")

        k_paths = 100
        source_not_found = HL_Standalone.copy()
        while len(source_not_found) != 0:

            # define a list to store path attributes for this iteration
            K_path_attributes = []

            # iterate through each "source_not_found" node only
            for src in HL_Standalone:
                for dest in HL_connected_nodes:
                    K_path_attributes = network.calculate_paths(subnetMatrix_HL, K_path_attributes, source = src, target = dest, k = k_paths)

            # Convert K_path_attributes list to a temporary DataFrame
            K_path_attributes_df = pd.DataFrame(K_path_attributes)

            # Optionally save to CSV (if you want to track progress)
            K_path_attributes_df.to_csv(file_name, index=False)

            # Sort by num_hops and distance
            K_path_attributes_df_sorted = K_path_attributes_df.groupby(['src_node'], group_keys=False).apply(
                lambda x: x.sort_values(['num_hops', 'distance'])
            )

            # find disjoint pairs from the sorted full dataset
            pairs_disjoint = network.land_pair_finder(HL_Standalone, K_path_attributes_df_sorted, num_pairs=1)

            # Update source_not_found to exclude newly matched source nodes
            source_not_found = np.setdiff1d(HL_Standalone, pairs_disjoint['src_node'].unique())

            print('Remaining src nodes: ', source_not_found)

            k_paths += 20
    
    file_name = results_dir/ f"{network.topology_name}_HL{hierarchy_level}_K_path_attributes_colocated.csv"

    if os.path.exists(file_name):

        print(f"Loading K-path attributes of HL{hierarchy_level} colocated...")

        K_path_attributes_colocated_df = pd.read_csv(file_name)
        K_path_attributes_colocated_df['links'] = K_path_attributes_colocated_df['links'].map(json.loads)
        K_path_attributes_colocated_df['nodes'] = K_path_attributes_colocated_df['nodes'].map(json.loads)
    else:

        print(f"Calculating K-path attributes of HL{hierarchy_level} colocated...")
        
        k_paths = 20

        # define a list to store path attributes
        K_path_attributes_colocated = []

        # iterate through each standalone HL4 node
        for src in HL_colocated:
            
            if (hierarchy_level == 3) and (src in HL2_Standalone):
                HL_connected_nodes_col = HL_connected_nodes - set(HL1_Standalone)
            else:
                HL_connected_nodes_col = HL_connected_nodes.copy()

            for dest in HL_connected_nodes:
                if src != dest:
                    K_path_attributes_colocated = network.calculate_paths(subnetMatrix_HL, K_path_attributes_colocated, source = src, target = dest, k = k_paths)

        # Convert K_path_attributes list to dataframe
        K_path_attributes_colocated_df = pd.DataFrame(K_path_attributes_colocated)

        # save dataframe to csv file
        K_path_attributes_colocated_df.to_csv(file_name, index = False)

    
    # sort dataframes based on num_hops and distance (in order)
    K_path_attributes_df_sorted = K_path_attributes_df.groupby(['src_node'], group_keys = False).apply(lambda x: x.sort_values(['num_hops', 'distance']))
    K_path_attributes_colocated_df_sorted = K_path_attributes_colocated_df.groupby(['src_node', 'dest_node'], group_keys = False).apply(lambda x: x.sort_values(['num_hops', 'distance']))

    # find disjoint pairs for standalone nodes
    pairs_disjoint = network.land_pair_finder(HL_Standalone, K_path_attributes_df_sorted, num_pairs = 1)

    print(f"process link GSNR of HL{hierarchy_level} ...")

    GSNR_opt_link, _, _, power_opt_link = c_band.process_link_gsnr(f_c_axis = f_c_axis, 
                                                                    Pch_dBm = Pch_dBm, 
                                                                    num_Ch_mat = num_Ch_mat,
                                                                    spectrum_C = spectrum_C,
                                                                    Nspan_array = np.ones(network.all_links.shape[0], dtype=int),
                                                                    hierarchy_level = hierarchy_level, 
                                                                    minimum_hierarchy_level = minimum_hierarchy_level, 
                                                                    result_directory = results_dir)
    
    planner.initialize_planner(num_fslots = num_fslots, 
                               hierarchy_level = hierarchy_level,
                               minimum_hierarchy_level = minimum_hierarchy_level,
                               SR = 170 * 1e9, # Symbol rate of 75G channels (in Hz)
                               Max_bit_rate_BVT = np.array([1000, 850, 700, 500, 350, 200]), # 1000 Gbps BVTs
                               Ref_license_capacity = np.array([200, 170, 140, 100, 70, 40]), # reference license capacities corresponding to BVTs
                               FP_max_num = 20, 
                               band_sepration_idx = [C_band_num_channels, 
                                                     C_band_num_channels + supC_band_num_channels])
    
    if hierarchy_level == minimum_hierarchy_level:

        # generate port capacity for HL4 nodes uisng Monte Carlo simulation
        planner.generate_initial_traffic_profile(num_nodes = len(HL_all),
                                                 optical_nodes = Optical_Nodes, 
                                                 monteCarlo_steps = 100,
                                                 min_rate = 20, # Gbps
                                                 max_rate = 200, # Gbps
                                                 seed = 50, 
                                                 result_directory = results_dir)

        # Traffic growth simulation over 10 years
        planner.simulate_traffic_annual(lowest_hierarchy_dict = hl_dict[f"HL{hierarchy_level}"], 
                                        CAGR = 0.4, 
                                        result_directory = results_dir)
        
    print(f"running planner for HL{hierarchy_level} ...")

    # run the planner for the current hierarchy level    
    planner.run_planner(hierarchy_level = hierarchy_level,
                        prev_hierarchy_level = prev_hierarchy_level,
                        pairs_disjoint = pairs_disjoint,
                        kpair_standalone = 1,
                        kpair_colocated = 1,
                        candidate_paths_standalone_df = K_path_attributes_df,
                        candidate_paths_colocated_df = K_path_attributes_colocated_df_sorted,
                        P_opt_links = power_opt_link,
                        GSNR_opt_link = GSNR_opt_link,
                        Nspan_array = np.ones(len(network.all_links)),
                        all_node_degree = all_node_degree,
                        minimum_level = minimum_hierarchy_level,
                        node_cap_update_idx = capacity_updt_index, 
                        result_directory = results_dir)


Processing hierarchy level: 4
Loading K-path attributes of HL4 ...
Loading K-path attributes of HL4 colocated...
process link GSNR of HL4 ...
Process Link GSNR: .....
Processing link 0 out of 166
Processing link 1 out of 166
Processing link 2 out of 166
Processing link 3 out of 166
Processing link 4 out of 166
Processing link 5 out of 166
Processing link 6 out of 166
Processing link 7 out of 166
Processing link 8 out of 166
Processing link 9 out of 166
Processing link 10 out of 166
Processing link 11 out of 166
Processing link 12 out of 166
Processing link 13 out of 166
Processing link 14 out of 166
Processing link 15 out of 166
Processing link 16 out of 166
Processing link 17 out of 166
Processing link 18 out of 166
Processing link 19 out of 166
Processing link 20 out of 166
Processing link 21 out of 166
Processing link 22 out of 166
Processing link 23 out of 166
Processing link 24 out of 166
Processing link 25 out of 166
Processing link 26 out of 166
Processing link 27 out of 166
Pro

In [50]:
analysing = analyse_result(network, 10, processing_level_list, results_dir, save_directory = results_dir)
analysing._load_data()
year = np.arange(1, 11)

In [51]:
cost_df_C_IRU_0v5 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_NoBypass_C_IRU_0v5", 
                    C_IRU = 0.5,
                    C_100GL_First = 2.25, 
                    C_100G_Added = 0.5625)

cost_df_C_IRU_0v01 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_NoBypass_C_IRU_0v01", 
                    C_IRU = 0.01,
                    C_100GL_First = 2.25, 
                    C_100G_Added = 0.5625)

cost_df_C_IRU_5 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_NoBypass_C_IRU_5", 
                    C_IRU = 5,
                    C_100GL_First = 2.25, 
                    C_100G_Added = 0.5625)

In [52]:
cost_df_C_IRU_0v01

,OPEX,Capex_RoB,Capex_MCS,Capex_100GL,CAPEX
0,49.0044,836.000000,38.850000,2259.000000,3133.850000
1,49.0044,0.000000,1.225000,319.500000,320.725000
2,49.0044,3.878280,3.531894,427.464337,434.874511
3,49.0044,10.720674,4.530004,656.416969,671.667646
4,49.0044,8.526676,8.061675,949.288589,965.876940
5,49.7272,8.848690,11.881862,1324.633948,1345.364499
6,52.3554,21.362787,22.094173,2076.719826,2120.176786
7,57.2408,27.053016,27.199180,2754.119928,2808.372124
8,65.1820,100.916506,39.834356,3886.450504,4027.201365
9,74.4270,133.567239,56.031986,5388.997484,5578.596709


In [53]:
node_capacity_profile_array_noBypass = np.load(results_dir / "MAN157_HL2_node_capacity_profile_array.npz")['node_capacity_profile_array']

In [54]:
node_capacity_profile_array_noBypass_HL4 = node_capacity_profile_array_noBypass[:, :, 3].cumsum(axis = 0)
node_capacity_profile_array_noBypass_HL3 = node_capacity_profile_array_noBypass[:, :, 2].cumsum(axis = 0)
node_capacity_profile_array_noBypass_HL2 = node_capacity_profile_array_noBypass[:, :, 1].cumsum(axis = 0)
node_capacity_profile_array_noBypass_HL1 = node_capacity_profile_array_noBypass[:, :, 0].cumsum(axis = 0)

ref_routers_cap = np.array([400, 800, 1600, 3200, 6400, 12800, 25600])
ref_routers_cost_metric = np.array([12, 24, 48, 96, 192, 384, 768]) # cost units in the comprehensive paper
ref_routers_cost_metric = ref_routers_cost_metric * 2000 / 15000 # cost unit 0f paper is 2000, our work is 15000
ref_routers_power_consumption = np.array([2000, 2500, 3100, 3900, 4900, 6100, 7600]) # in Watts
ref_year = 5 # year five is the reference for selection of router

In [55]:
# NoBypass Scenario
HL4_SA_num_routers_noBypass = []
HL4_SA_selected_router_noBypass = []

HL4_co_num_routers_noBypass = []
HL4_co_selected_router_noBypass = []

HL3_SA_num_routers_noBypass = []
HL3_SA_selected_router_noBypass = []

HL3_co_num_routers_noBypass = []
HL3_co_selected_router_noBypass = []

HL2_SA_num_routers_noBypass = []
HL2_SA_selected_router_noBypass = []

HL2_co_num_routers_noBypass = []
HL2_co_selected_router_noBypass = []

HL1_SA_num_routers_noBypass = []
HL1_SA_selected_router_noBypass = []

routers_tracker_array_noBypass = np.zeros(shape = (len(ref_routers_cap), len(year)), dtype = np.int64)

for hl4_sa in hl_dict['HL4']['standalone']:
    ref_traffic = node_capacity_profile_array_noBypass_HL4[ref_year - 1, hl4_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL4_SA_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL4[:, hl4_sa] // closest_bigger_router_cap).astype(int) + 1
    HL4_SA_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL4[:, hl4_sa] // closest_bigger_router_cap + 1)

for hl4_co in hl_dict['HL4']['colocated']:
    ref_traffic = node_capacity_profile_array_noBypass_HL4[ref_year - 1, hl4_co]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL4_co_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL4[:, hl4_co] // closest_bigger_router_cap).astype(int) + 1
    HL4_co_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL4[:, hl4_co] // closest_bigger_router_cap + 1)

for hl3_sa in hl_dict['HL3']['standalone']:
    ref_traffic = node_capacity_profile_array_noBypass_HL3[ref_year - 1, hl3_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL3_SA_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL3[:, hl3_sa] // closest_bigger_router_cap).astype(int) + 1
    HL3_SA_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL3[:, hl3_sa] // closest_bigger_router_cap + 1)

for hl3_co in hl_dict['HL3']['colocated']:
    ref_traffic = node_capacity_profile_array_noBypass_HL3[ref_year - 1, hl3_co]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL3_co_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL3[:, hl3_co] // closest_bigger_router_cap).astype(int) + 1
    HL3_co_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL3[:, hl3_co] // closest_bigger_router_cap + 1)

for hl2_sa in hl_dict['HL2']['standalone']:
    ref_traffic = node_capacity_profile_array_noBypass_HL2[ref_year - 1, hl2_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL2_SA_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL2[:, hl2_sa] // closest_bigger_router_cap).astype(int) + 1
    HL2_SA_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL2[:, hl2_sa] // closest_bigger_router_cap + 1)

for hl2_co in hl_dict['HL2']['colocated']:
    ref_traffic = node_capacity_profile_array_noBypass_HL2[ref_year - 1, hl2_co]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL2_co_selected_router_noBypass.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL2[:, hl2_co] // closest_bigger_router_cap).astype(int) + 1
    HL2_co_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL2[:, hl2_co] // closest_bigger_router_cap + 1)
    
for hl1_sa in hl_dict['HL1']['standalone']:
	ref_traffic = node_capacity_profile_array_noBypass_HL1[ref_year - 1, hl1_sa]
	idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
	closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
	HL1_SA_selected_router_noBypass.append(closest_bigger_router_cap)
	idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
	routers_tracker_array_noBypass[idx, :] += (node_capacity_profile_array_noBypass_HL1[:, hl1_sa] // closest_bigger_router_cap).astype(int) + 1
	HL1_SA_num_routers_noBypass.append(node_capacity_profile_array_noBypass_HL1[:, hl1_sa] // closest_bigger_router_cap + 1)
    
num_routers_HL4_SA_total_yearly_noBypass = np.sum(HL4_SA_num_routers_noBypass, axis = 0)
num_routers_HL4_co_total_yearly_noBypass = np.sum(HL4_co_num_routers_noBypass, axis = 0)
num_routers_HL3_SA_total_yearly_noBypass = np.sum(HL3_SA_num_routers_noBypass, axis = 0)
num_routers_HL3_co_total_yearly_noBypass = np.sum(HL3_co_num_routers_noBypass, axis = 0)
num_routers_HL2_SA_total_yearly_noBypass = np.sum(HL2_SA_num_routers_noBypass, axis = 0)
num_routers_HL2_co_total_yearly_noBypass = np.sum(HL2_co_num_routers_noBypass, axis = 0)
num_routers_HL1_SA_total_yearly_noBypass = np.sum(HL1_SA_num_routers_noBypass, axis = 0)

total_num_router_yearly_noBypass = num_routers_HL4_SA_total_yearly_noBypass + num_routers_HL4_co_total_yearly_noBypass + \
    + num_routers_HL3_SA_total_yearly_noBypass + num_routers_HL3_co_total_yearly_noBypass \
        + num_routers_HL2_SA_total_yearly_noBypass + num_routers_HL2_co_total_yearly_noBypass \
			+ num_routers_HL1_SA_total_yearly_noBypass

In [56]:
# np.savez(results_dir / "MAN157_routers_tracker_array_noBypass.npz", routers_tracker_array = routers_tracker_array_noBypass)

In [57]:
# NoBypass Scenario
HL4_SA_cost_noBypass = []
HL4_co_cost_noBypass = []
HL3_SA_cost_noBypass = []
HL3_co_cost_noBypass = []
HL2_SA_cost_noBypass = []
HL2_co_cost_noBypass = []
HL1_SA_cost_noBypass = []

HL4_SA_power_noBypass = []
HL4_co_power_noBypass = []
HL3_SA_power_noBypass = []
HL3_co_power_noBypass = []
HL2_SA_power_noBypass = []
HL1_SA_cost_noBypass = []
HL2_co_cost_noBypass = []

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL4_SA_selected_router_noBypass]
HL4_SA_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL4_SA_noBypass_power = ref_routers_power_consumption[indices]
HL4_SA_cost_noBypass = [ai * bi for ai, bi in zip(HL4_SA_num_routers_noBypass, HL4_SA_noBypass_cost_metrics)]
HL4_SA_power_noBypass = [ai * bi for ai, bi in zip(HL4_SA_num_routers_noBypass, HL4_SA_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL4_co_selected_router_noBypass]
HL4_co_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL4_co_noBypass_power = ref_routers_power_consumption[indices]
HL4_co_cost_noBypass = [ai * bi for ai, bi in zip(HL4_co_num_routers_noBypass, HL4_co_noBypass_cost_metrics)]
HL4_co_power_noBypass = [ai * bi for ai, bi in zip(HL4_co_num_routers_noBypass, HL4_co_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL3_SA_selected_router_noBypass]
HL3_SA_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL3_SA_noBypass_power = ref_routers_power_consumption[indices]
HL3_SA_cost_noBypass = [ai * bi for ai, bi in zip(HL3_SA_num_routers_noBypass, HL3_SA_noBypass_cost_metrics)]
HL3_SA_power_noBypass = [ai * bi for ai, bi in zip(HL3_SA_num_routers_noBypass, HL3_SA_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL3_co_selected_router_noBypass]
HL3_co_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL3_co_noBypass_power = ref_routers_power_consumption[indices]
HL3_co_cost_noBypass = [ai * bi for ai, bi in zip(HL3_co_num_routers_noBypass, HL3_co_noBypass_cost_metrics)]
HL3_co_power_noBypass = [ai * bi for ai, bi in zip(HL3_co_num_routers_noBypass, HL3_co_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL2_SA_selected_router_noBypass]
HL2_SA_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL2_SA_noBypass_power = ref_routers_power_consumption[indices]
HL2_SA_cost_noBypass = [ai * bi for ai, bi in zip(HL2_SA_num_routers_noBypass, HL2_SA_noBypass_cost_metrics)]
HL2_SA_power_noBypass = [ai * bi for ai, bi in zip(HL2_SA_num_routers_noBypass, HL2_SA_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL2_co_selected_router_noBypass]
HL2_co_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL2_co_noBypass_power = ref_routers_power_consumption[indices]
HL2_co_cost_noBypass = [ai * bi for ai, bi in zip(HL2_co_num_routers_noBypass, HL2_co_noBypass_cost_metrics)]
HL2_co_power_noBypass = [ai * bi for ai, bi in zip(HL2_co_num_routers_noBypass, HL2_co_noBypass_power)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL1_SA_selected_router_noBypass]
HL1_SA_noBypass_cost_metrics = ref_routers_cost_metric[indices]
HL1_SA_noBypass_power = ref_routers_power_consumption[indices]
HL1_SA_cost_noBypass = [ai * bi for ai, bi in zip(HL1_SA_num_routers_noBypass, HL1_SA_noBypass_cost_metrics)]
HL1_SA_power_noBypass = [ai * bi for ai, bi in zip(HL1_SA_num_routers_noBypass, HL1_SA_noBypass_power)]

IP_total_cost_noBypass = np.sum(HL4_SA_cost_noBypass, axis = 0) + np.sum(HL4_co_cost_noBypass, axis = 0) + \
    np.sum(HL3_SA_cost_noBypass, axis = 0) + np.sum(HL3_co_cost_noBypass, axis = 0) \
        + np.sum(HL2_SA_cost_noBypass, axis = 0) \
            + np.sum(HL2_co_cost_noBypass, axis = 0) + np.sum(HL1_SA_cost_noBypass, axis = 0)
            
IP_total_power_noBypass = np.sum(HL4_SA_power_noBypass, axis = 0) + np.sum(HL4_co_power_noBypass, axis = 0) + \
    np.sum(HL3_SA_power_noBypass, axis = 0) + np.sum(HL3_co_power_noBypass, axis = 0) \
        + np.sum(HL2_SA_power_noBypass, axis = 0) \
            + np.sum(HL2_co_power_noBypass, axis = 0) + np.sum(HL1_SA_power_noBypass, axis = 0)

In [58]:
OTCO_annual = (cost_df_C_IRU_0v5['OPEX'] + cost_df_C_IRU_0v5['CAPEX']).values
TCO_Final_year = (IP_total_cost_noBypass[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(55137.22558126199)

In [59]:
OTCO_annual = (cost_df_C_IRU_0v01['OPEX'] + cost_df_C_IRU_0v01['CAPEX']).values
TCO_Final_year = (IP_total_cost_noBypass[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(28483.459981261985)

In [60]:
OTCO_annual = (cost_df_C_IRU_5['OPEX'] + cost_df_C_IRU_5['CAPEX']).values
TCO_Final_year = (IP_total_cost_noBypass[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(299916.705581262)

In [61]:
# np.savez(results_dir / "MAN157_IP_Cost_Power_noBypass.npz", 
#          IP_Cost_Cum = IP_total_cost_noBypass, 
#          IP_Power_Cum = IP_total_power_noBypass 
#          )

In [62]:
power_unit_ROADM = 72 #910 # Watt
power_unit_MCS = 10 # Watt
typ_power_400G_BVT = 290 # Watt

In [63]:
Total_BVT_noBypass = analysing.bvt_data['HL4']['HL_BVTNum_All'].sum(axis = 1) + analysing.bvt_data['HL3']['HL_BVTNum_All'].sum(axis = 1) +\
        analysing.bvt_data['HL2']['HL_BVTNum_All'].sum(axis = 1)

In [64]:
degree_links_cum = analysing.link_data['HL2']['HL_CDegree_Domain'].sum(axis = 1) +\
    analysing.link_data['HL2']['HL_SuperCDegree_Domain'].sum(axis = 1) +\
        analysing.link_data['HL2']['HL_LDegree_Domain'].sum(axis = 1)

In [65]:
np.diff(Total_BVT_noBypass, prepend=0)

array([ 888.,   28.,   80.,  100.,  176.,  260.,  492.,  600.,  884.,
       1252.])

In [66]:
# Energy Consumption Between scenarios
energy_consumption_annual_noBypass = (np.diff(Total_BVT_noBypass, prepend=0) * typ_power_400G_BVT +\
    np.diff(degree_links_cum, prepend=0) * power_unit_ROADM +\
        np.diff(Total_BVT_noBypass, prepend=0) / 16 * power_unit_MCS) * 365 * 24 / 1e6


total_served_traffic_noBypass = node_capacity_profile_array_noBypass_HL4.sum(axis = 1)
IP_energy_consumption_annual_noBypass = np.diff(IP_total_power_noBypass * 365 * 24 / 1e6, prepend=0)

# np.savez(results_dir / 'MAN157_energy_optical_electrical_noBypass.npz',
#          optical_energy_annual = energy_consumption_annual_noBypass,
#          IP_energy_annual = IP_energy_consumption_annual_noBypass, 
#          total_served_traffic = total_served_traffic_noBypass)

ENERGY_Final_Year = (energy_consumption_annual_noBypass + IP_energy_consumption_annual_noBypass).cumsum()[-1]
ENERGY_Final_Year

np.float64(33104.94228)